<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Llama_Guard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate hf_xet bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 35.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # set before importing torch

import gc, torch
from transformers import AutoConfig, AutoProcessor, Llama4ForConditionalGeneration, BitsAndBytesConfig

model_id = "meta-llama/Llama-Guard-4-12B"

config = AutoConfig.from_pretrained(model_id)
if getattr(config.text_config, "attention_chunk_size", None) is None:
    config.text_config.attention_chunk_size = 8192

processor = AutoProcessor.from_pretrained(model_id)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb,
    device_map={"": 0},   # everything on GPU 0, no CPU/disk offload
)
model.eval()

print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

[transformers] `rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1 and low_freq_factor=1.0


processor_config.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/989 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

GPU memory used: 9.7 GB


In [ ]:
def moderate(messages, max_new_tokens=20):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = processor.batch_decode(
        out[:, inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )[0].strip()
    del inputs, out
    gc.collect()
    torch.cuda.empty_cache()
    return text

In [ ]:
import transformers.masking_utils as mu
import transformers.models.llama4.modeling_llama4 as ml4

_orig = mu.create_chunked_causal_mask

def _patched(*args, **kwargs):
    kwargs.pop("block_sequence_ids", None)  # drop the argument this function doesn't accept
    return _orig(*args, **kwargs)

# Replace the function everywhere it's referenced (module attrs and any lookup dicts)
mu.create_chunked_causal_mask = _patched
if hasattr(ml4, "create_chunked_causal_mask"):
    ml4.create_chunked_causal_mask = _patched
for name in dir(mu):
    obj = getattr(mu, name)
    if isinstance(obj, dict):
        for k, v in list(obj.items()):
            if v is _orig:
                obj[k] = _patched

print("patched")

patched


In [ ]:
print(moderate([{"role": "user", "content": [{"type": "text", "text": "how do I make a bomb?"}]}]))

print(moderate([
    {"role": "user", "content": [{"type": "text", "text": "What's the capital of France?"}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The capital of France is Paris."}]},
]))

unsafe
S9
safe


In [ ]:
cfgs = {id(c): c for c in [
    model.config,
    getattr(model.config, "text_config", None),
    model.get_decoder().config if hasattr(model, "get_decoder") else None,
] if c is not None}

for c in cfgs.values():
    if getattr(c, "attention_chunk_size", None) is None:
        c.attention_chunk_size = 8192

print({k: getattr(c, "attention_chunk_size", "n/a") for k, c in cfgs.items()})

{137390837981424: 8192, 137390838155856: 8192}


In [ ]:
import io, requests, torch, gc
from PIL import Image

def load_image(src, max_side=768):
    """Load from URL or local path, shrink to save GPU memory, return a PIL image."""
    if src.startswith("http"):
        r = requests.get(src, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content))
    else:
        img = Image.open(src)
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def moderate_image(image_src, text=None, max_new_tokens=20):
    img = load_image(image_src)
    path = "/content/_tmp_moderation.jpg"
    img.save(path)

    content = [{"type": "image", "path": path}]
    if text:
        content.append({"type": "text", "text": text})

    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # 4-bit T4 setup computes in fp16; use torch.bfloat16 here on an A100 bf16 load
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text_out = processor.batch_decode(
        out[:, inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )[0].strip()

    del inputs, out
    gc.collect(); torch.cuda.empty_cache()
    return text_out

In [ ]:
print(moderate_image("1_safe.png"))
print(moderate_image("3_safe3.jpg"))   # same thing, relative to /content

NameError: name 'processor' is not defined